# Reproducing the Philadelphia Tax Year 2027 assessment report

This notebook rebuilds the statistics published on Fair Measure's Tax Year 2027 report from the project's local, versioned data products. It is both a reproduction and a release audit: cells stop with an assertion error if the source data, public JSON, or reported results disagree.

**What this is:** an early comparison of the 2026 and 2027 assessments against the same independent model estimate dated July 14, 2026.

**What this is not:** a Tax Year 2027 sales-ratio study. The new assessments take effect January 1, 2027, so post-effective-date sales do not yet exist. The model is an imperfect benchmark, not ground truth. Race is used only after valuation as neighborhood-level audit context and is never a model input.

## Running the notebook

From the repository root, run `uv sync`, build the public-data pipeline through `uv run fair-measure export-web-stats` as described in the repository README, then open this file with `uv run --with jupyterlab jupyter lab notebooks/ty2027_report_reproduction.ipynb`.

Required local artifacts are `data/staged/assessments.parquet`, `data/marts/assessment_screen.parquet`, `data/marts/assessment_features.parquet`, the latest `acs_tract_demographics` snapshot, and `web/src/data/siteStats.json`. The data lake is gitignored because of its size; every source is public and the CLI records snapshot manifests and fetch times.

## Official timing and methodology

The City says the new assessments take effect January 1, 2027 and mailed notices beginning June 29, 2026. OPA's methodology summary says its valuation used sales from January 2020 through June 2025 and time-adjusted them to the appraisal date. The City released its own and an external ratio study on July 8, 2026.

Sources: [City release](https://www.phila.gov/2026-06-30-city-of-philadelphia-to-mail-2027-property-assessments-and-launch-expanded-outreach-to-connect-homeowners-to-tax-relief/), [OPA methodology summary](https://www.phila.gov/media/20260629163818/opa-tax-year-2027-mass-appraisal-valuation-methodology-summary.pdf), and [annual ratio studies](https://www.phila.gov/documents/annual-ratio-studies/).

In [ ]:
from pathlib import Path
import json

import numpy as np
import polars as pl

from philly_fair_measure import config
from philly_fair_measure.diagnostics.acs_sensitivity import _acs_frame, join_tracts
from philly_fair_measure.diagnostics.annual_roll import (
    AnnualRollConfig,
    build_annual_roll_report,
)
from philly_fair_measure.models.metrics import ratio_metrics, vertical_equity_indicator

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Run this notebook from inside the repository.'

DATA = config.data_dir()
if not DATA.is_absolute():
    DATA = ROOT / DATA
PUBLISHED_PATH = ROOT / 'web' / 'src' / 'data' / 'siteStats.json'
published = json.loads(PUBLISHED_PATH.read_text())['annual_report']

TAX_YEAR = int(published['tax_year'])
COMPARISON_YEAR = int(published['comparison_year'])
DIRECTION_DEADBAND = float(published['correction']['direction_threshold_pct']) / 100

def check(actual: float, expected: float, tolerance: float = 0.051) -> None:
    assert np.isclose(actual, expected, atol=tolerance, rtol=0), (actual, expected)

print(f'Repository: {ROOT}')
print(f'Data lake: {DATA}')
print(f'Published comparison: TY {COMPARISON_YEAR} → TY {TAX_YEAR}')

## 1. Load and validate the matched assessment rolls

The public report covers modeled single-family residential properties with positive values in both assessment years. Records with a source-data warning still receive an estimate, but are excluded from fairness direction and ratio metrics.

In [ ]:
assessments = pl.read_parquet(DATA / 'staged' / 'assessments.parquet')
screen = pl.read_parquet(DATA / 'marts' / 'assessment_screen.parquet')

two_years = assessments.filter(
    pl.col('year_parsed').is_in([COMPARISON_YEAR, TAX_YEAR])
    & (pl.col('market_value') > 0)
).select('parcel_number', 'year_parsed', 'market_value')
duplicate_keys = two_years.group_by('parcel_number', 'year_parsed').len().filter(pl.col('len') > 1)
assert duplicate_keys.is_empty(), 'Duplicate parcel-year assessment records found.'

year_values = (
    two_years.pivot(on='year_parsed', index='parcel_number', values='market_value')
    .rename({str(COMPARISON_YEAR): 'old_assessment', str(TAX_YEAR): 'new_assessment'})
)

matched = (
    screen.filter(
        (pl.col('model_family') == 'residential')
        & (pl.col('opa_market_value') > 0)
        & (pl.col('display_median') > 0)
    )
    .select(
        'parcel_id', 'loc_zip5', 'opa_market_value', 'display_median',
        'display_pi_low_90', 'display_pi_high_90', 'record_quality_warning',
    )
    .join(year_values, left_on='parcel_id', right_on='parcel_number', how='inner')
    .drop_nulls(['old_assessment', 'new_assessment', 'display_median'])
    .with_columns(
        pl.col('record_quality_warning').fill_null(True).alias('_data_warning'),
        (pl.col('new_assessment') / pl.col('old_assessment') - 1).alias('_change'),
        (pl.col('old_assessment') / pl.col('display_median')).alias('_old_ratio'),
        (pl.col('new_assessment') / pl.col('display_median')).alias('_new_ratio'),
        (pl.col('old_assessment') / pl.col('display_median')).log().abs().alias('_old_error'),
        (pl.col('new_assessment') / pl.col('display_median')).log().abs().alias('_new_error'),
    )
    .with_columns(
        pl.when(pl.col('_data_warning')).then(pl.lit('data_warning'))
        .when(pl.col('_new_error') < pl.col('_old_error') - DIRECTION_DEADBAND).then(pl.lit('corrective'))
        .when(pl.col('_new_error') > pl.col('_old_error') + DIRECTION_DEADBAND).then(pl.lit('widening'))
        .otherwise(pl.lit('no_clear_change')).alias('_direction'),
        (pl.col('old_assessment').is_between(pl.col('display_pi_low_90'), pl.col('display_pi_high_90'))).alias('_old_inside'),
        (pl.col('new_assessment').is_between(pl.col('display_pi_low_90'), pl.col('display_pi_high_90'))).alias('_new_inside'),
    )
)

assert matched.height == int(published['n_properties'])
assert matched.filter(pl.col('opa_market_value') != pl.col('new_assessment')).is_empty(), (
    'The assessment displayed on the property screen does not match the TY2027 history.'
)
reliable = matched.filter(~pl.col('_data_warning'))
assert reliable.height == int(published['correction']['n_reliable'])
print(f'{matched.height:,} matched homes; {reliable.height:,} included in fairness metrics.')
print('PASS — unique parcel-year keys and an exact TY2027 screen/history match.')

## 2. Reproduce the citywide change and movement toward the benchmark

A move is called corrective or widening only when its absolute log-distance from the model estimate changes by more than two percentage points. Smaller changes are reported as no clear change.

In [ ]:
roll_row = matched.select(
    pl.col('old_assessment').median().alias('old_median'),
    pl.col('new_assessment').median().alias('new_median'),
    pl.col('_change').median().alias('median_change'),
    pl.col('_change').quantile(0.1).alias('p10_change'),
    pl.col('_change').quantile(0.9).alias('p90_change'),
    pl.col('old_assessment').sum().alias('old_total'),
    pl.col('new_assessment').sum().alias('new_total'),
    (pl.col('new_assessment') > pl.col('old_assessment')).mean().alias('increased'),
    (pl.col('new_assessment') < pl.col('old_assessment')).mean().alias('decreased'),
    (pl.col('new_assessment') == pl.col('old_assessment')).mean().alias('unchanged'),
).to_dicts()[0]
computed_roll = {
    'old_median_usd': round(roll_row['old_median']),
    'new_median_usd': round(roll_row['new_median']),
    'median_change_pct': round(roll_row['median_change'] * 100, 1),
    'p10_change_pct': round(roll_row['p10_change'] * 100, 1),
    'p90_change_pct': round(roll_row['p90_change'] * 100, 1),
    'total_change_pct': round((roll_row['new_total'] / roll_row['old_total'] - 1) * 100, 1),
    'increased_pct': round(roll_row['increased'] * 100, 1),
    'decreased_pct': round(roll_row['decreased'] * 100, 1),
    'unchanged_pct': round(roll_row['unchanged'] * 100, 1),
}
for key, value in computed_roll.items():
    check(value, float(published['roll'][key]))

direction_counts = {row['_direction']: row['len'] for row in reliable.group_by('_direction').len().to_dicts()}
n_reliable = reliable.height
inside = reliable.select(
    pl.col('_old_inside').mean().alias('old_inside'),
    pl.col('_new_inside').mean().alias('new_inside'),
).to_dicts()[0]
computed_correction = {
    'data_warning_pct': round((matched.height - n_reliable) / matched.height * 100, 1),
    'corrective_pct': round(direction_counts.get('corrective', 0) / n_reliable * 100, 1),
    'widening_pct': round(direction_counts.get('widening', 0) / n_reliable * 100, 1),
    'no_clear_pct': round(direction_counts.get('no_clear_change', 0) / n_reliable * 100, 1),
    'old_inside_interval_pct': round(inside['old_inside'] * 100, 1),
    'new_inside_interval_pct': round(inside['new_inside'] * 100, 1),
}
for key, value in computed_correction.items():
    check(value, float(published['correction'][key]))

print(pl.DataFrame([computed_roll]))
print(pl.DataFrame([computed_correction]))
print('PASS — citywide change, warning share, direction, and interval coverage match the website.')

## 3. Reproduce vertical-equity results

Homes are ranked by the independent model estimate and split into five equal groups. Using one fixed benchmark prevents either assessment year's values from deciding which price group a home enters. A ratio above 100% means the assessment is above the model estimate; a ratio below 100% means it is below the model estimate.

In [ ]:
tiered = reliable.with_columns(
    (((pl.col('display_median').rank(method='ordinal') - 1) * 5 / pl.len()).floor().cast(pl.Int8) + 1).alias('_tier')
)
tier_table = (
    tiered.group_by('_tier').agg(
        pl.len().alias('n'),
        (pl.col('_old_ratio').median() * 100).round(1).alias('old_ratio_pct'),
        (pl.col('_new_ratio').median() * 100).round(1).alias('new_ratio_pct'),
        (pl.col('_change').median() * 100).round(1).alias('median_change_pct'),
    ).sort('_tier')
)
for actual, expected in zip(tier_table.to_dicts(), published['tiers'], strict=True):
    assert actual['_tier'] == expected['tier'] and actual['n'] == expected['n']
    for key in ('old_ratio_pct', 'new_ratio_pct', 'median_change_pct'):
        check(float(actual[key]), float(expected[key]))

benchmark = reliable['display_median'].to_numpy().astype(float)
old = reliable['old_assessment'].to_numpy().astype(float)
new = reliable['new_assessment'].to_numpy().astype(float)
old_ratio_metrics, new_ratio_metrics = ratio_metrics(old, benchmark), ratio_metrics(new, benchmark)
old_vei = vertical_equity_indicator(old, benchmark).vei
new_vei = vertical_equity_indicator(new, benchmark).vei
computed_metrics = {
    'old_prd': round(old_ratio_metrics.prd, 4), 'new_prd': round(new_ratio_metrics.prd, 4),
    'old_prb': round(old_ratio_metrics.prb, 4), 'new_prb': round(new_ratio_metrics.prb, 4),
    'old_vei': round(old_vei, 1), 'new_vei': round(new_vei, 1),
    'old_cod': round(old_ratio_metrics.cod, 1), 'new_cod': round(new_ratio_metrics.cod, 1),
}
for key, path in {
    'old_prd': ('vertical_equity', 'prd', 'old'), 'new_prd': ('vertical_equity', 'prd', 'new'),
    'old_prb': ('vertical_equity', 'prb', 'old'), 'new_prb': ('vertical_equity', 'prb', 'new'),
    'old_vei': ('vertical_equity', 'vei', 'old'), 'new_vei': ('vertical_equity', 'vei', 'new'),
    'old_cod': ('uniformity', 'old_cod'), 'new_cod': ('uniformity', 'new_cod'),
}.items():
    expected = published
    for part in path:
        expected = expected[part]
    check(float(computed_metrics[key]), float(expected), tolerance=0.000051 if 'pr' in key else 0.051)

print(tier_table)
print(pl.DataFrame([computed_metrics]))
print('PASS — all five tiers and COD, PRD, PRB, and VEI match the website.')

### The public quintile chart

This is the report's main figure, generated directly from the verified tier table above. The connecting segment shows how each value group's assessment ratio changed from Tax Year 2026 to Tax Year 2027.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plot_rows = tier_table.sort('_tier').to_dicts()
plot_labels = [row['label'] for row in published['tiers']]
plot_y = np.arange(len(plot_rows))[::-1]
old_values = np.array([row['old_ratio_pct'] for row in plot_rows], dtype=float)
new_values = np.array([row['new_ratio_pct'] for row in plot_rows], dtype=float)

fig, ax = plt.subplots(figsize=(10, 5.4), constrained_layout=True)
ink, muted, blue, connector = '#17253b', '#5c6b7d', '#175a97', '#9eb9d2'

for y, old_value, new_value in zip(plot_y, old_values, new_values, strict=True):
    ax.plot([old_value, new_value], [y, y], color=connector, linewidth=3, solid_capstyle='round', zorder=1)
ax.scatter(old_values, plot_y, s=58, facecolor='white', edgecolor=blue, linewidth=2, zorder=3, label=f'{COMPARISON_YEAR} assessment')
ax.scatter(new_values, plot_y, s=58, facecolor=blue, edgecolor=blue, linewidth=1, zorder=4, label=f'{TAX_YEAR} assessment')

for y, old_value, new_value in zip(plot_y, old_values, new_values, strict=True):
    left, right = sorted([old_value, new_value])
    ax.text(left - 1.2, y, f'{left:.0f}%', ha='right', va='center', color=muted, fontsize=9)
    ax.text(right + 1.2, y, f'{right:.0f}%', ha='left', va='center', color=ink, fontsize=9, fontweight='bold')

ax.axvline(100, color=muted, linewidth=1)
ax.annotate('Below our estimate', xy=(84, len(plot_rows) - 0.42), xytext=(99, len(plot_rows) - 0.42), ha='right', va='bottom', color=muted, fontsize=9, arrowprops={'arrowstyle': '->', 'color': muted, 'linewidth': 1})
ax.annotate('Above our estimate', xy=(116, len(plot_rows) - 0.42), xytext=(101, len(plot_rows) - 0.42), ha='left', va='bottom', color=muted, fontsize=9, arrowprops={'arrowstyle': '->', 'color': muted, 'linewidth': 1})
ax.set_xlim(70, 155)
ax.set_ylim(-0.65, len(plot_rows) - 0.15)
ax.set_yticks(plot_y, plot_labels)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:.0f}%'))
ax.set_xlabel('OPA assessment as a share of the independent estimate', color=ink, labelpad=10)
ax.set_title('The assessment gap widened between cheaper and more expensive homes', loc='left', color=ink, fontweight='bold', pad=22)
ax.grid(axis='x', color='#dfe5eb', linewidth=0.8)
ax.set_axisbelow(True)
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_color('#b8c1cb')
ax.tick_params(axis='y', length=0, colors=ink)
ax.tick_params(axis='x', colors=muted)
ax.legend(frameon=False, loc='lower right', ncols=2, labelcolor=ink, fontsize=9, handletextpad=0.6)
fig.text(0.01, -0.015, 'Five equal-sized groups ranked by the model estimate; records with data-quality warnings are excluded.', color=muted, fontsize=9)
plt.show()

## Appendix: neighborhood-race context and audit exclusions

The labels below describe the majority racial composition of a Census tract, not the race of a homeowner. The public summary shows the three prespecified large groups. This cell also prints every other category and the unmatched count so coverage is explicit.

In [ ]:
locations = pl.read_parquet(
    DATA / 'marts' / 'assessment_features.parquet',
    columns=['parcel_id', 'loc_lon', 'loc_lat'],
).unique('parcel_id')
demographics = join_tracts(locations, _acs_frame(DATA)).select('parcel_id', 'acs_majority_race')
with_race = matched.join(demographics, on='parcel_id', how='left').with_columns(
    pl.col('acs_majority_race').fill_null('Unmatched').alias('_race_group')
)
coverage = (
    with_race.group_by('_race_group').agg(
        pl.len().alias('n_all'),
        (~pl.col('_data_warning')).sum().alias('n_reliable'),
        (pl.col('_data_warning').mean() * 100).round(1).alias('warning_pct'),
    ).sort('n_all', descending=True)
)
print('All tract categories and unmatched properties:')
print(coverage)

race_reliable = with_race.filter(~pl.col('_data_warning'))
race_table = (
    race_reliable.filter(pl.col('_race_group').is_in([row['group'] for row in published['race_context']]))
    .group_by('_race_group').agg(
        pl.len().alias('n'),
        (pl.col('_old_ratio').median() * 100).round(1).alias('old_ratio_pct'),
        (pl.col('_new_ratio').median() * 100).round(1).alias('new_ratio_pct'),
        (pl.col('_change').median() * 100).round(1).alias('median_change_pct'),
    )
)
race_by_name = {row['_race_group']: row for row in race_table.to_dicts()}
for expected in published['race_context']:
    actual = race_by_name[expected['group']]
    assert actual['n'] == expected['n']
    for key in ('old_ratio_pct', 'new_ratio_pct', 'median_change_pct'):
        check(float(actual[key]), float(expected[key]))

race_all_records = (
    with_race.filter(pl.col('_race_group').is_in([row['group'] for row in published['race_context']]))
    .group_by('_race_group').agg(
        pl.len().alias('n'),
        (pl.col('_old_ratio').median() * 100).round(1).alias('old_ratio_pct'),
        (pl.col('_new_ratio').median() * 100).round(1).alias('new_ratio_pct'),
    ).sort('_race_group')
)
warning_by_value = (
    matched.with_columns(
        (((pl.col('display_median').rank(method='ordinal') - 1) * 5 / pl.len()).floor().cast(pl.Int8) + 1).alias('value_fifth')
    ).group_by('value_fifth').agg(
        pl.len().alias('n'),
        (pl.col('_data_warning').mean() * 100).round(1).alias('warning_pct'),
    ).sort('value_fifth')
)
print('Published race rows:')
print(race_table.sort('_race_group'))
print('Sensitivity including data-warning records (not used for the published verdict):')
print(race_all_records)
print('Warning rate by modeled-value fifth:')
print(warning_by_value)
print('PASS — published race rows match; omitted categories and warning rates are shown above.')

## Appendix: race comparison among similarly valued homes

Neighborhood race and home value are strongly related in Philadelphia. A raw table cannot say whether the racial pattern is distinct from the citywide price pattern. This sensitivity check divides all reliable homes into 20 equal groups by model-estimated value, keeps the common-support groups with at least 500 homes from each reported race category, calculates each category's median log ratio inside each value group, and gives every retained value group equal weight. It is descriptive, not causal, but it prevents a difference in group price distributions or a handful of extreme-value homes from doing all the work.

In [ ]:
target_groups = [row['group'] for row in published['race_context']]
value_binned = (
    reliable.with_columns(
        (((pl.col('display_median').rank(method='ordinal') - 1) * 20 / pl.len()).floor().cast(pl.Int8) + 1).alias('_value_bin'),
        pl.col('_old_ratio').log().alias('_old_log_ratio'),
        pl.col('_new_ratio').log().alias('_new_log_ratio'),
    )
    .join(demographics, on='parcel_id', how='left')
)
targeted = value_binned.filter(pl.col('acs_majority_race').is_in(target_groups))
within_bin = targeted.group_by('acs_majority_race', '_value_bin').agg(
    pl.len().alias('n'),
    pl.col('_old_log_ratio').median().alias('old_log_median'),
    pl.col('_new_log_ratio').median().alias('new_log_median'),
)
common_bins = (
    within_bin.group_by('_value_bin').agg(
        pl.col('acs_majority_race').n_unique().alias('groups'),
        pl.col('n').min().alias('smallest_group'),
    ).filter((pl.col('groups') == len(target_groups)) & (pl.col('smallest_group') >= 500))
    .sort('_value_bin')['_value_bin'].to_list()
)
assert len(common_bins) >= 5, 'Too little common value support for a stable sensitivity check.'
within_bin = within_bin.filter(pl.col('_value_bin').is_in(common_bins))
bin_coverage = within_bin.group_by('acs_majority_race').agg(
    pl.col('_value_bin').n_unique().alias('bins'), pl.col('n').min().alias('smallest_cell')
)
assert bin_coverage['bins'].min() == len(common_bins)

standardized = (
    within_bin.group_by('acs_majority_race').agg(
        pl.col('old_log_median').mean().alias('_old_standardized_log'),
        pl.col('new_log_median').mean().alias('_new_standardized_log'),
    ).with_columns(
        (pl.col('_old_standardized_log').exp() * 100).round(1).alias('old_value_adjusted_pct'),
        (pl.col('_new_standardized_log').exp() * 100).round(1).alias('new_value_adjusted_pct'),
    ).select('acs_majority_race', 'old_value_adjusted_pct', 'new_value_adjusted_pct')
    .sort('acs_majority_race')
)
old_adjusted_spread = float(standardized['old_value_adjusted_pct'].max() - standardized['old_value_adjusted_pct'].min())
new_adjusted_spread = float(standardized['new_value_adjusted_pct'].max() - standardized['new_value_adjusted_pct'].min())

city_bin = value_binned.group_by('_value_bin').agg(
    pl.col('_old_log_ratio').median().alias('_city_old_log'),
    pl.col('_new_log_ratio').median().alias('_city_new_log'),
)
peer_residual = (
    targeted.filter(pl.col('_value_bin').is_in(common_bins)).join(city_bin, on='_value_bin', how='left')
    .with_columns(
        (pl.col('_old_log_ratio') - pl.col('_city_old_log')).alias('_old_residual'),
        (pl.col('_new_log_ratio') - pl.col('_city_new_log')).alias('_new_residual'),
    ).group_by('acs_majority_race').agg(
        ((pl.col('_old_residual').median().exp() - 1) * 100).round(1).alias('old_vs_same_value_city_pct'),
        ((pl.col('_new_residual').median().exp() - 1) * 100).round(1).alias('new_vs_same_value_city_pct'),
    ).sort('acs_majority_race')
)

print(f'Common-support value bins: {common_bins}')
print('Cell coverage:')
print(bin_coverage.sort('acs_majority_race'))
print('Equal-value-distribution standardized ratios:')
print(standardized)
print(f'Adjusted max–min gap: {old_adjusted_spread:.1f} points → {new_adjusted_spread:.1f} points')
print('Difference from citywide homes in the same modeled-value bin (positive means assessed higher):')
print(peer_residual)
print('Sensitivity result:', 'the adjusted gap widened' if new_adjusted_spread > old_adjusted_spread else 'the adjusted gap did not widen')

def standardization_robustness(n_bins: int, minimum_cell: int) -> dict[str, object]:
    frame = (
        reliable.with_columns(
            (((pl.col('display_median').rank(method='ordinal') - 1) * n_bins / pl.len()).floor().cast(pl.Int16) + 1).alias('_bin'),
            pl.col('_old_ratio').log().alias('_old_log'),
            pl.col('_new_ratio').log().alias('_new_log'),
        ).join(demographics, on='parcel_id', how='left')
        .filter(pl.col('acs_majority_race').is_in(target_groups))
    )
    cells = frame.group_by('acs_majority_race', '_bin').agg(
        pl.len().alias('n'),
        pl.col('_old_log').median().alias('old'),
        pl.col('_new_log').median().alias('new'),
    )
    overlap = (
        cells.group_by('_bin').agg(
            pl.col('acs_majority_race').n_unique().alias('groups'),
            pl.col('n').min().alias('smallest'),
        ).filter((pl.col('groups') == len(target_groups)) & (pl.col('smallest') >= minimum_cell))['_bin'].to_list()
    )
    standardized_rows = (
        cells.filter(pl.col('_bin').is_in(overlap)).group_by('acs_majority_race').agg(
            (pl.col('old').mean().exp() * 100).alias('old'),
            (pl.col('new').mean().exp() * 100).alias('new'),
        ).to_dicts()
    )
    values = {row['acs_majority_race']: row for row in standardized_rows}
    old_values = [float(row['old']) for row in standardized_rows]
    new_values = [float(row['new']) for row in standardized_rows]
    return {
        'bins': n_bins, 'minimum_cell': minimum_cell, 'overlap_bins': len(overlap),
        'old_gap_pp': round(max(old_values) - min(old_values), 1),
        'new_gap_pp': round(max(new_values) - min(new_values), 1),
        'white_moved': 'closer' if abs(values['White alone']['new'] - 100) < abs(values['White alone']['old'] - 100) else 'farther',
        'black_moved': 'closer' if abs(values['Black alone']['new'] - 100) < abs(values['Black alone']['old'] - 100) else 'farther',
        'hispanic_moved': 'closer' if abs(values['Hispanic/Latino, any race']['new'] - 100) < abs(values['Hispanic/Latino, any race']['old'] - 100) else 'farther',
    }

robustness = pl.DataFrame([
    standardization_robustness(n_bins, minimum_cell)
    for n_bins, minimum_cell in [(10, 200), (10, 500), (20, 200), (20, 500), (20, 1000), (40, 200), (40, 500)]
])
assert (robustness['new_gap_pp'] > robustness['old_gap_pp']).all()
print('Robustness across bin counts and common-support thresholds:')
print(robustness)

### Current sensitivity result

The raw rows match the website: majority-White tracts move from 92.4% to 96.5% of the model estimate, majority-Black tracts from 104.5% to 107.0%, and majority-Hispanic tracts from 112.3% to 119.1%.

That direction is not stable after value adjustment. Within the primary common modeled-value range, the standardized ratios are 105.7% to 109.3% for majority-White tracts, 97.7% to 99.0% for majority-Black tracts, and 99.1% to 102.8% for majority-Hispanic tracts. The adjusted maximum gap still widens, from 8.0 to 10.3 percentage points, but the White and Black directions reverse. Across seven reasonable bin and minimum-cell choices, the adjusted gap widens every time; White and Hispanic tracts move farther from the benchmark every time, while the Black-tract direction is mixed.

Takeaway: neighborhood-race differences remain, but the public raw table is substantially entangled with the cheaper-versus-expensive-home pattern. It should not be presented as a race-specific or causal result without showing this value-adjusted sensitivity.

## Final integration check

The earlier cells independently rebuild the statistics. This final integration check calls the production report generator and requires every published field other than the unused boundary-transect appendix to match exactly.

In [ ]:
reproduced = build_annual_roll_report(
    assessments,
    screen,
    AnnualRollConfig(
        tax_year=TAX_YEAR,
        comparison_year=COMPARISON_YEAR,
        effective_date=published['effective_date'],
        sales_cutoff=published['sales_cutoff'],
        min_log_direction=DIRECTION_DEADBAND,
    ),
    demographics=demographics,
)
published_without_transect = {key: value for key, value in published.items() if key != 'boundary_transect'}
# ZIP rows are sorted by change; tied rows may arrive in either order. Compare their content by ZIP.
reproduced['areas'] = sorted(reproduced['areas'], key=lambda row: row['zip'])
published_without_transect['areas'] = sorted(published_without_transect['areas'], key=lambda row: row['zip'])
assert reproduced == published_without_transect
print('PASS — the production generator exactly reproduces the complete published TY2027 contract.')

## Interpretation guardrails

- These results support statements about agreement with this project's model benchmark, not definitive over- or under-assessment.
- A citywide improvement in model agreement can coexist with worse vertical equity. Report both.
- The neighborhood-race table is descriptive. Even after value standardization, it does not identify race as the cause.
- Nineteen percent of matched properties carry a known data-quality warning and are excluded from fairness metrics. Review the printed exclusion rates before publication.
- Replace this early benchmark check with post-effective-date sales evidence when enough valid 2027 market sales are available.